In [ ]:
!module li

In [ ]:
import sys, os, re
import numpy as np
import matplotlib.pyplot as plt
import openpmd_api as io

### Iterate through raw directories and get all sims

### Load saved DS meta data

In [ ]:
loaded = np.load("selfFocusDataSet.npz", allow_pickle=True)
refKeys = loaded["refKeys"]
allRuns = loaded["allRuns"]

In [ ]:
nParams = len(refKeys)

In [ ]:
samples = np.empty((len(allRuns), nParams+1))
for i, r in enumerate(allRuns):
    for ik, k in enumerate(refKeys):
        samples[i, ik] = r["params"][k]
    samples[i, -1] = r["dirI"]
print(samples.shape)

In [ ]:
samples = []
RE_iteration = re.compile(".*iteration_([0-9]+).*")
col = 7
for r in allRuns:
    p = r["path"].split('/')
    if p[col] != 'active_learning':
        continue
    m = RE_iteration.match(p[col+1])
    if not m:
        continue
    row = np.concatenate((list(r["params"][k] for k in refKeys) , [float(m[1])]))
    samples.append(row)
samples = np.stack(samples)
print(samples.shape)

In [ ]:
nPlots = nParams - 1
fig, axs = plt.subplots(nPlots, nPlots)
for x in range(nPlots):
    for y in range(x+1, nParams):
        ax = axs[y-1, x]
        ax.scatter(samples[:, x], samples[:, y], s=1, c=samples[:,-1])
        if y < nParams-1:
            ax.xaxis.set_ticklabels([])
        else:
            ax.set_xlabel(refKeys[x])
        if x > 0:
            ax.yaxis.set_ticklabels([])
        else:
            ax.set_ylabel(refKeys[y])
    for y in range(1, x+1):
        ax = axs[y-1, x]
        ax.set_visible(False)

In [ ]:
openPMDPath = "simOutput/openPMD"
series = io.Series(os.path.join(allRuns[0]["path"], openPMDPath, "simData_%T.bp"), io.Access.read_only)

In [ ]:
!ls /trinity/shared/pkg/filelib/adios/2.9.2-cuda121/gcc/12.2.0/openmpi/4.1.5-cuda121-blosc2-py3122/lib/python3.12

In [ ]:
it = series.iterations[20000]
t = list(series.iterations.items())[0][1]
t.attributes

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(20, 10))
Ecomps = {}
for comp in "xyz":
    eh = it.meshes["E"][comp]
    Ecomps[comp] = np.zeros(eh.shape, dtype=np.float32)
    eh.load_chunk(Ecomps[comp])
    series.flush()
    Ecomps[comp] = Ecomps[comp][0]

for i, comp in enumerate("xyz"):
    el = Ecomps[comp]    
    ax = axs[i]
    ax.set_title(f"$E_{comp}$")
    im = ax.imshow(el)
    plt.colorbar(im, ax=ax)
fig.savefig("Efields.png")
fig.show()

In [ ]:
plt.imshow(Ecomps["x"] - Ecomps["z"])
plt.colorbar()